# 02 - Construcción de subconjuntos Yelp para traducción al español

## Objetivo

Este notebook construye subconjuntos estratégicos del corpus Yelp Food Reviews para traducirlos al español y utilizarlos en fases posteriores de IA/NLP de Hidden Gems.

No se traduce todo el corpus completo, ya que sería costoso y podría introducir ruido artificial. En su lugar, se generan subconjuntos controlados:

1. **Balanced subset**: subconjunto equilibrado por sentimiento.
2. **Food-rich subset**: reseñas con alta probabilidad de contener menciones de platos.
3. **Gold seed subset**: subconjunto pequeño y prioritario para traducción, revisión y futura anotación manual.

## Uso posterior

Estos subconjuntos permitirán:

- adaptar modelos al español;
- comparar inglés vs español traducido;
- preparar ejemplos para detección de platos;
- construir un dataset oro inicial;
- acercar el entrenamiento al caso real de Sevilla.

In [1]:
# ============================================================
# 01. Imports y configuración
# ============================================================

from pathlib import Path
import json
import random
import re
import hashlib
import warnings
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 250)

print("Entorno inicializado correctamente.")

Entorno inicializado correctamente.


In [3]:
# ============================================================
# 02. Montar Google Drive de forma robusta
# ============================================================

from pathlib import Path

USE_DRIVE = True

if USE_DRIVE:
    try:
        from google.colab import drive

        # Limpieza preventiva del punto de montaje
        !fusermount -u /content/drive 2>/dev/null || true
        !rm -rf /content/drive

        drive.mount("/content/drive", force_remount=True, timeout_ms=120000)

        PROJECT_DIR = Path("/content/drive/MyDrive/hidden_gems_ai")
        DATA_DIR = PROJECT_DIR / "data"

        PROJECT_DIR.mkdir(parents=True, exist_ok=True)
        DATA_DIR.mkdir(parents=True, exist_ok=True)

        print("Google Drive montado correctamente.")
        print("PROJECT_DIR:", PROJECT_DIR)
        print("DATA_DIR:", DATA_DIR)

    except Exception as e:
        print("No se ha podido montar Google Drive.")
        print("Error:", e)
        print("Se usará almacenamiento temporal de Colab en /content/hidden_gems_ai")

        USE_DRIVE = False
        PROJECT_DIR = Path("/content/hidden_gems_ai")
        DATA_DIR = PROJECT_DIR / "data"

        PROJECT_DIR.mkdir(parents=True, exist_ok=True)
        DATA_DIR.mkdir(parents=True, exist_ok=True)

else:
    PROJECT_DIR = Path("/content/hidden_gems_ai")
    DATA_DIR = PROJECT_DIR / "data"

    PROJECT_DIR.mkdir(parents=True, exist_ok=True)
    DATA_DIR.mkdir(parents=True, exist_ok=True)

# Directorios específicos del notebook
OUTPUT_DIR = PROJECT_DIR / "outputs" / "translation_subsets"
CHUNKS_DIR = OUTPUT_DIR / "chunks_for_translation"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHUNKS_DIR.mkdir(parents=True, exist_ok=True)

print("Ruta activa del proyecto:", PROJECT_DIR)
print("Ruta activa de datos:", DATA_DIR)
print("Ruta activa de outputs:", OUTPUT_DIR)
print("Ruta activa de chunks:", CHUNKS_DIR)

No se ha podido montar Google Drive.
Error: mount failed
Se usará almacenamiento temporal de Colab en /content/hidden_gems_ai
Ruta activa del proyecto: /content/hidden_gems_ai
Ruta activa de datos: /content/hidden_gems_ai/data
Ruta activa de outputs: /content/hidden_gems_ai/outputs/translation_subsets
Ruta activa de chunks: /content/hidden_gems_ai/outputs/translation_subsets/chunks_for_translation


In [4]:
# ============================================================
# 03. Ruta del corpus Yelp NLP
# ============================================================

CORPUS_PATH = DATA_DIR / "yelp_food_reviews_corpus_sample_100k_lines.jsonl"

if not CORPUS_PATH.exists():
    raise FileNotFoundError(
        f"No se ha encontrado el corpus en:\n{CORPUS_PATH}\n\n"
        "Sube el archivo yelp_food_reviews_corpus_sample_100k_lines.jsonl "
        "a la carpeta data del proyecto en Google Drive."
    )

print("Corpus encontrado:")
print(CORPUS_PATH)

Corpus encontrado:
/content/hidden_gems_ai/data/yelp_food_reviews_corpus_sample_100k_lines.jsonl


In [5]:
# ============================================================
# 04. Cargar corpus JSONL
# ============================================================

def load_jsonl(path: Path) -> pd.DataFrame:
    records = []
    invalid_lines = 0

    with path.open("r", encoding="utf-8") as f:
        for line in tqdm(f, desc="Leyendo JSONL"):
            line = line.strip()
            if not line:
                continue

            try:
                records.append(json.loads(line))
            except json.JSONDecodeError:
                invalid_lines += 1

    print(f"Registros cargados: {len(records):,}")
    print(f"Líneas inválidas: {invalid_lines:,}")

    return pd.DataFrame(records)


df_raw = load_jsonl(CORPUS_PATH)

print("Shape:", df_raw.shape)
display(df_raw.head(3))

Leyendo JSONL: 0it [00:00, ?it/s]

Registros cargados: 79,270
Líneas inválidas: 0
Shape: (79270, 20)


,corpus_document_id,source_system_code,source_dataset,source_entity_type,source_review_id,source_business_id,source_user_id,text,text_normalized,language,rating_value,sentiment_label_from_rating,review_date,corpus_split,task_scope,is_training_eligible,quality_flags,business_metadata,source_metrics,created_at
0,yelp_2fbfd094613536a7b8c9231b,yelp_open_dataset,yelp_open_dataset,yelp_review,KU_O5udG6zpxOg-VcAEodg,XQfwVwDr-v0ZS3_CbbE5Xw,mh_-eMZ6K5RLWhZyISBhwA,"If you decide to eat here, just be aware it is going to take about 2 hours from beginning to end. We have tried it multiple times, because I want to like it! I have been to it's other locations in NJ and never had a bad experience. The food is go...","If you decide to eat here, just be aware it is going to take about 2 hours from beginning to end. We have tried it multiple times, because I want to like it! I have been to it's other locations in NJ and never had a bad experience. The food is go...",en,3.0,neutral,2018-07-07 22:09:11,train,"[dish_extraction, review_sentiment, dish_sentiment, recommendation_signal]",True,"{'has_text': True, 'has_rating': True, 'has_business_metadata': True, 'text_length_chars': 511, 'label_is_weak': True, 'label_source': 'rating_value'}","{'business_name': 'Turning Point of North Wales', 'city': 'North Wales', 'state': 'PA', 'stars_business': 3.0, 'review_count_business': 169, 'is_open': 1, 'categories_list': ['Restaurants', 'Breakfast & Brunch', 'Food', 'Juice Bars & Smoothies', ...","{'useful_count': 0, 'funny_count': 0, 'cool_count': 0}",2026-05-06T10:57:56.623430+00:00
1,yelp_94c5a64cecd4448d105e5c8a,yelp_open_dataset,yelp_open_dataset,yelp_review,saUsX_uimxRlCVr67Z4Jig,YjUWPpI6HXG530lwP-fb2A,8g_iMtfSiwikVnbP2etR0A,"Family diner. Had the buffet. Eclectic assortment: a large chicken leg, fried jalapeño, tamale, two rolled grape leaves, fresh melon. All good. Lots of Mexican choices there. Also has a menu with breakfast served all day long. Friendly, attentive...","Family diner. Had the buffet. Eclectic assortment: a large chicken leg, fried jalapeño, tamale, two rolled grape leaves, fresh melon. All good. Lots of Mexican choices there. Also has a menu with breakfast served all day long. Friendly, attentive...",en,3.0,neutral,2014-02-05 20:30:30,train,"[dish_extraction, review_sentiment, dish_sentiment, recommendation_signal]",True,"{'has_text': True, 'has_rating': True, 'has_business_metadata': True, 'text_length_chars': 339, 'label_is_weak': True, 'label_source': 'rating_value'}","{'business_name': 'Kettle Restaurant', 'city': 'Tucson', 'state': 'AZ', 'stars_business': 3.5, 'review_count_business': 47, 'is_open': 1, 'categories_list': ['Restaurants', 'Breakfast & Brunch'], 'food_category_tags': ['Breakfast & Brunch', 'Rest...","{'useful_count': 0, 'funny_count': 0, 'cool_count': 0}",2026-05-06T10:57:56.624430+00:00
2,yelp_69e10d25d69774ab39af6571,yelp_open_dataset,yelp_open_dataset,yelp_review,AqPFMleE6RsU23_auESxiA,kxX2SOes4o-D3ZQBkiMRfA,_7bHUi9Uuf5__HHc_Q8guQ,"Wow! Yummy, different, delicious. Our favorite is the lamb curry and korma. With 10 different kinds of naan!!! Don't let the outside deter you (because we almost changed our minds)...go in and try something new! You'll be glad you did!","Wow! Yummy, different, delicious. Our favorite is the lamb curry and korma. With 10 different kinds of naan!!! Don't let the outside deter you (because we almost changed our minds)...go in and try something new! You'll be glad you did!",en,5.0,positive,2015-01-04 00:01:03,train,"[dish_extraction, review_sentiment, dish_sentiment, recommendation_signal]",True,"{'has_text': True, 'has_rating': True, 'has_business_metadata': True, 'text_length_chars': 235, 'label_is_weak': True, 'label_source': 'rating_value'}","{'business_name': 'Zaika', 'city': 'Philadelphia', 'state': 'PA', 'stars_business': 4.0, 'review_count_business': 181, 'is_open': 1, 'categories_list': ['Halal', 'Pakistani', 'Restaurants', 'Indian'], 'food_category_tags': ['Indian', 'Restaurant

In [6]:
# ============================================================
# 05. Validación inicial del corpus
# ============================================================

required_cols = [
    "corpus_document_id",
    "source_review_id",
    "source_business_id",
    "text",
    "text_normalized",
    "language",
    "rating_value",
    "sentiment_label_from_rating",
    "corpus_split",
]

missing_cols = [col for col in required_cols if col not in df_raw.columns]

if missing_cols:
    raise ValueError(f"Faltan columnas obligatorias: {missing_cols}")

print("Columnas obligatorias presentes.")

print("\nDistribución de idioma:")
display(df_raw["language"].value_counts())

print("\nDistribución de splits:")
display(df_raw["corpus_split"].value_counts())

print("\nDistribución de sentimiento:")
display(df_raw["sentiment_label_from_rating"].value_counts())

print("\nNulos en columnas clave:")
display(df_raw[required_cols].isna().sum())

Columnas obligatorias presentes.

Distribución de idioma:


,count
language,
en,79270



Distribución de splits:


,count
corpus_split,
train,63540
test,7881
validation,7849



Distribución de sentimiento:


,count
sentiment_label_from_rating,
positive,54857
negative,14578
neutral,9835



Nulos en columnas clave:


,0
corpus_document_id,0
source_review_id,0
source_business_id,0
text,0
text_normalized,0
language,0
rating_value,0
sentiment_label_from_rating,0
corpus_split,0


In [7]:
# ============================================================
# 06. Preparar dataframe base
# ============================================================

df = df_raw.copy()

df["text_en"] = (
    df["text"]
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

df["text_en_normalized"] = (
    df["text_normalized"]
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

df["sentiment_label"] = (
    df["sentiment_label_from_rating"]
    .astype(str)
    .str.lower()
    .str.strip()
)

df["split"] = (
    df["corpus_split"]
    .astype(str)
    .str.lower()
    .str.strip()
)

df["rating_value"] = pd.to_numeric(df["rating_value"], errors="coerce")
df["text_length_chars"] = df["text_en"].str.len()

valid_labels = {"positive", "neutral", "negative"}
valid_splits = {"train", "validation", "test"}

df = df[
    df["sentiment_label"].isin(valid_labels)
    & df["split"].isin(valid_splits)
    & df["text_en"].notna()
    & (df["text_length_chars"] > 0)
].copy()

print("Shape tras preparación:", df.shape)

display(
    pd.crosstab(df["split"], df["sentiment_label"])
)

Shape tras preparación: (79270, 25)


sentiment_label,negative,neutral,positive
split,,,
test,1428,936,5517
train,11701,7894,43945
validation,1449,1005,5395


In [8]:
# ============================================================
# 07. Extraer metadata de negocio
# ============================================================

def parse_maybe_json(value):
    if isinstance(value, dict):
        return value
    if isinstance(value, str):
        try:
            parsed = json.loads(value)
            if isinstance(parsed, dict):
                return parsed
        except Exception:
            return {}
    return {}


def safe_list(value):
    if isinstance(value, list):
        return value
    if isinstance(value, str):
        try:
            parsed = json.loads(value)
            if isinstance(parsed, list):
                return parsed
        except Exception:
            pass

        return [part.strip() for part in value.split(",") if part.strip()]

    return []


business_names = []
cities = []
states = []
food_category_tags = []
business_stars = []
business_review_counts = []

for value in df.get("business_metadata", [{}]):
    bm = parse_maybe_json(value)

    business_names.append(
        bm.get("business_name")
        or bm.get("name")
        or bm.get("source_name_raw")
        or ""
    )

    cities.append(bm.get("city") or "")
    states.append(bm.get("state") or "")

    tags = (
        bm.get("food_category_tags")
        or bm.get("categories_list")
        or bm.get("categories")
        or []
    )
    food_category_tags.append(safe_list(tags))

    business_stars.append(bm.get("stars") or bm.get("business_stars") or None)
    business_review_counts.append(bm.get("review_count") or bm.get("business_review_count") or None)

df["business_name"] = business_names
df["city"] = cities
df["state"] = states
df["food_category_tags"] = food_category_tags
df["business_stars"] = business_stars
df["business_review_count"] = business_review_counts

display(
    df[
        [
            "corpus_document_id",
            "business_name",
            "city",
            "state",
            "sentiment_label",
            "split",
            "food_category_tags",
            "text_en"
        ]
    ].head(5)
)

,corpus_document_id,business_name,city,state,sentiment_label,split,food_category_tags,text_en
0,yelp_2fbfd094613536a7b8c9231b,Turning Point of North Wales,North Wales,PA,neutral,train,"[American (New), Breakfast & Brunch, Coffee & Tea, Food, Juice Bars & Smoothies, Restaurants, Sandwiches]","If you decide to eat here, just be aware it is going to take about 2 hours from beginning to end. We have tried it multiple times, because I want to like it! I have been to it's other locations in NJ and never had a bad experience. The food is go..."
1,yelp_94c5a64cecd4448d105e5c8a,Kettle Restaurant,Tucson,AZ,neutral,train,"[Breakfast & Brunch, Restaurants]","Family diner. Had the buffet. Eclectic assortment: a large chicken leg, fried jalapeño, tamale, two rolled grape leaves, fresh melon. All good. Lots of Mexican choices there. Also has a menu with breakfast served all day long. Friendly, attentive..."
2,yelp_69e10d25d69774ab39af6571,Zaika,Philadelphia,PA,positive,train,"[Indian, Restaurants]","Wow! Yummy, different, delicious. Our favorite is the lamb curry and korma. With 10 different kinds of naan!!! Don't let the outside deter you (because we almost changed our minds)...go in and try something new! You'll be glad you did!"
3,yelp_95095495afc3a77ce68723a2,Melt,New Orleans,LA,positive,train,"[American (Traditional), Bars, Food, Restaurants, Sandwiches]","Cute interior and owner (?) gave us tour of upcoming patio/rooftop area which will be great on beautiful days like today. Cheese curds were very good and very filling. Really like that sandwiches come w salad, esp after eating too many curds! Had..."
4,yelp_9d31d4621975f4080301a6c8,Dmitri's,Philadelphia,PA,negative,train,"[Greek, Mediterranean, Restaurants, Seafood]",I am a long term frequent customer of this establishment. I just went in to order take out (3 apps) and was told they're too busy to do it. Really? The place is maybe half full at best. Does your dick reach your ass? Yes? Go fuck yourself! I'm a ...


In [9]:
# ============================================================
# 08. Diccionario inicial de términos gastronómicos en inglés
# ============================================================

FOOD_TERMS_EN = [
    # General food / dishes
    "food", "dish", "dishes", "meal", "menu", "appetizer", "appetizers",
    "starter", "entree", "main course", "dessert", "desserts",

    # Common American / international dishes
    "burger", "burgers", "cheeseburger", "cheeseburgers",
    "pizza", "pizzas", "sandwich", "sandwiches",
    "fries", "french fries", "home fries", "wings", "chicken wings",
    "fried chicken", "ribs", "ribeye", "steak", "steaks",
    "brisket", "pulled pork", "bbq", "barbecue",
    "hot dog", "hot dogs", "sonoran dog",

    # Italian
    "pasta", "penne", "spaghetti", "lasagna", "lasagne",
    "gnocchi", "marinara", "eggplant", "baked eggplant",
    "veal milanese", "ravioli", "risotto",

    # Mexican / Latin
    "taco", "tacos", "burrito", "burritos", "quesadilla", "quesadillas",
    "enchilada", "enchiladas", "tamale", "tamales", "salsa",
    "guacamole", "nachos",

    # Asian
    "sushi", "sashimi", "ramen", "noodles", "pho",
    "curry", "lamb curry", "korma", "naan",
    "dumpling", "dumplings", "tempura", "edamame",
    "salmon", "tuna", "tuna tacos",

    # Breakfast / brunch
    "breakfast", "brunch", "waffle", "waffles", "pancake", "pancakes",
    "french toast", "omelet", "omelette", "eggs benedict",
    "bacon", "eggs",

    # Seafood
    "seafood", "crab", "crab legs", "crab cake", "crab cakes",
    "oyster", "oysters", "calamari", "shrimp", "lobster",

    # Soups / salads
    "salad", "salads", "soup", "soups", "hummus",

    # Desserts / drinks
    "cake", "cakes", "pie", "pumpkin pie", "ice cream",
    "custard", "frozen custard", "latte", "coffee", "tea",
]

FOOD_TERMS_EN = sorted(set(term.lower() for term in FOOD_TERMS_EN), key=len, reverse=True)

print("Número de términos gastronómicos iniciales:", len(FOOD_TERMS_EN))
print(FOOD_TERMS_EN[:30])

Número de términos gastronómicos iniciales: 115
['frozen custard', 'baked eggplant', 'cheeseburgers', 'fried chicken', 'chicken wings', 'eggs benedict', 'veal milanese', 'french fries', 'french toast', 'cheeseburger', 'pulled pork', 'main course', 'sonoran dog', 'quesadillas', 'pumpkin pie', 'home fries', 'appetizers', 'enchiladas', 'quesadilla', 'crab cakes', 'sandwiches', 'tuna tacos', 'lamb curry', 'guacamole', 'spaghetti', 'breakfast', 'enchilada', 'crab legs', 'appetizer', 'crab cake']


In [10]:
# ============================================================
# 09. Calcular food-rich score
# ============================================================

food_pattern = re.compile(
    r"\b(" + "|".join(re.escape(term) for term in FOOD_TERMS_EN) + r")\b",
    flags=re.IGNORECASE
)

context_patterns = [
    r"\bhad the\b",
    r"\bordered\b",
    r"\btried\b",
    r"\bloved\b",
    r"\bfavorite\b",
    r"\brecommend\b",
    r"\bserved with\b",
    r"\bcame with\b",
    r"\bwas delicious\b",
    r"\bwas amazing\b",
    r"\bwas good\b",
    r"\bwas bad\b",
    r"\bwas bland\b",
    r"\bwas cold\b",
    r"\bwas dry\b",
]

context_regex = re.compile("|".join(context_patterns), flags=re.IGNORECASE)


def extract_food_terms(text: str) -> list[str]:
    if not isinstance(text, str):
        return []
    matches = food_pattern.findall(text)
    return sorted(set(match.lower() for match in matches))


def compute_food_rich_score(row) -> int:
    text = row["text_en"]

    terms = extract_food_terms(text)
    term_score = len(terms)

    context_score = len(context_regex.findall(text))

    tags = row.get("food_category_tags", [])
    tag_score = min(len(tags), 5)

    # Dar más peso a términos concretos encontrados en el texto.
    score = (term_score * 3) + (context_score * 2) + tag_score

    return int(score)


df["candidate_food_terms_found"] = df["text_en"].apply(extract_food_terms)
df["candidate_food_term_count"] = df["candidate_food_terms_found"].apply(len)
df["food_rich_score"] = df.apply(compute_food_rich_score, axis=1)

print("Resumen food_rich_score:")
display(df["food_rich_score"].describe())

display(
    df.sort_values("food_rich_score", ascending=False)[
        [
            "sentiment_label",
            "split",
            "rating_value",
            "food_rich_score",
            "candidate_food_terms_found",
            "text_en"
        ]
    ].head(10)
)

Resumen food_rich_score:


,food_rich_score
count,79270.000000
mean,11.753274
std,7.278969
min,1.000000
25%,7.000000
50%,10.000000
75%,15.000000
max,88.000000


,sentiment_label,split,rating_value,food_rich_score,candidate_food_terms_found,text_en
45901,positive,train,5.0,88,"[appetizer, cake, calamari, dessert, dishes, eggplant, entree, food, ice cream, main course, meal, menu, salad, salmon, seafood, soup, soups, steak]","This review is for their restaurant week selection. WOW! So good. 4 course meal (plus bread) for $35 each, plus tax and tip. If I had to do it again, I'd choose the Tarte de Legumes, the Saumon, and the Tart Tatin. Also, good-sized portions on ev..."
58804,positive,train,5.0,87,"[appetizer, appetizers, cake, dessert, dish, entree, food, gnocchi, ice cream, lobster, meal, menu, pie, salad, salmon, seafood, soup, steaks, tuna]",RW review: I had the BEST time last night at Square 1682 for RW festivities! We had a reservation for 7:30pm for party of 6. My husband and I arrived a few minutes early and were greeted by a sleek hostess immediately and even took our coats. We ...
61855,positive,train,5.0,87,"[appetizers, bacon, bbq, brunch, cake, crab, crab cake, dishes, eggplant, eggs, food, fries, menu, oysters, pasta, pizza, pizzas, sandwich, sandwiches, seafood, shrimp]","I have been patronizing Katie's since early 1999, back in my days of being employed by the City of NOLA. It is one of my absolute favorite restaurants in the city and has never left me anything but completely satiated after every visit. It is als..."
8209,neutral,train,3.0,78,"[bacon, brisket, cake, coffee, curry, dessert, dish, dishes, eggplant, food, ice cream, meal, menu, pasta, risotto, salad, soup, tempura]","My husband, daughter, and I dined at Sbraga tonight, excited to try Top Chef winner Kevin Sbraga's fare. We're all educated foodies and 2 of 3 of us like aggressively seasoned food. From our reservation through receiving our check, the service wa..."
53591,positive,train,4.0,71,"[bacon, bbq, brisket, brunch, burger, cheeseburger, chicken wings, dish, dishes, food, menu, nachos, pulled pork, salad, salsa, sandwich, soup, wings]","Khyber Pass Pub is located in a historic city, in a historic building, with a historic bar that came from the Centennial World's Fair if 1876. Khyber Pass itself, has only been open since 1970, making it a baby in comparison. Still, the place is ..."
62242,positive,train,5.0,68,"[appetizer, appetizers, bacon, burger, burgers, cake, entree, food, fried chicken, fries, guacamole, main course, meal, menu, noodles, salad, salads]","I have seen various reviews on Tavern. Some very negative. Others slightly positive. Most complimented the food but nothing spectacular. So, this past Friday night a few friends and I got together for dinner and I wanted to try some place new. I ..."
37574,positive,train,5.0,66,"[brisket, chicken wings, dessert, dishes, food, fried chicken, ice cream, menu, noodles, pho, shrimp, soup, soups, tacos, tea, waffle, waffles]",One of my favorites in town! I've tried almost everything on the menu... and love it alllllll! These are my favorite dishes: *Starters Shrimp and Pork Rolls Pork Belly Tacos STICKY CHICKEN WINGS *Soups Wonton Soup Pho Brisket add Meatballs Tofu &...
1125,positive,train,4.0,64,"[appetizers, bacon, burger, crab, dessert, dishes, entree, food, fries, meal, menu, shrimp, starter]","If you are looking to be impressed by French-German dishes in New Orleans, please go to Luke Restaurant, I know I was. The restaurant is located in the Central Business District, next to the Hilton and in the former Masonic Temple Building, so th..."
24253,negative,train,2.0,63,"[appetizer, dish, food, menu, salad, salmon, sashimi, shrimp, soup, sushi, tea, tuna]","Remember my review on Sengyo?? I just rather go there or try other sushi places for sushi instead of Oh Yoko. But before I get into that: First off, I've ordered Pork Katsu as take-out twice here before - once for lunch (I think it was $11) and o..."
35649,positive,train,5.0,63,"[bacon, breakfast, cake, dessert, dishes, edamame, entree, gnocchi, lobster, meal, pie, salad, soup, steak]","Still 

In [11]:
# ============================================================
# 10. Funciones de muestreo
# ============================================================

def sample_by_plan(
    source_df: pd.DataFrame,
    plan: dict,
    sort_by: str | None = None,
    ascending: bool = False,
    random_state: int = RANDOM_STATE
) -> pd.DataFrame:
    """
    plan esperado:
    {
      "train": {"positive": 100, "neutral": 100, "negative": 100},
      "validation": {...},
      "test": {...}
    }
    """

    samples = []
    shortages = []

    for split_name, label_plan in plan.items():
        for label_name, n in label_plan.items():
            subset = source_df[
                (source_df["split"] == split_name)
                & (source_df["sentiment_label"] == label_name)
            ].copy()

            available = len(subset)

            if available == 0:
                shortages.append(
                    {
                        "split": split_name,
                        "label": label_name,
                        "requested": n,
                        "available": 0,
                        "selected": 0,
                    }
                )
                continue

            selected_n = min(n, available)

            if sort_by is not None:
                subset = subset.sort_values(sort_by, ascending=ascending)
                selected = subset.head(selected_n)
            else:
                selected = subset.sample(
                    n=selected_n,
                    random_state=random_state
                )

            samples.append(selected)

            if selected_n < n:
                shortages.append(
                    {
                        "split": split_name,
                        "label": label_name,
                        "requested": n,
                        "available": available,
                        "selected": selected_n,
                    }
                )

    if not samples:
        return pd.DataFrame(), shortages

    result = pd.concat(samples, ignore_index=True)
    return result, shortages


def show_subset_summary(subset_df: pd.DataFrame, subset_name: str):
    print("=" * 100)
    print(subset_name)
    print("=" * 100)
    print("Total:", len(subset_df))

    print("\nPor split:")
    display(subset_df["split"].value_counts())

    print("\nPor sentimiento:")
    display(subset_df["sentiment_label"].value_counts())

    print("\nSplit x sentimiento:")
    display(pd.crosstab(subset_df["split"], subset_df["sentiment_label"]))

    print("\nFood rich score:")
    display(subset_df["food_rich_score"].describe())

In [12]:
# ============================================================
# 11. Crear balanced subset 6.000
# ============================================================

# 6.000 reviews:
# - 2.000 por clase
# - por clase: 1.600 train, 200 validation, 200 test

balanced_plan = {
    "train": {
        "positive": 1600,
        "neutral": 1600,
        "negative": 1600,
    },
    "validation": {
        "positive": 200,
        "neutral": 200,
        "negative": 200,
    },
    "test": {
        "positive": 200,
        "neutral": 200,
        "negative": 200,
    },
}

balanced_df, balanced_shortages = sample_by_plan(
    df,
    balanced_plan,
    sort_by=None
)

balanced_df["subset_balanced_6000"] = True

show_subset_summary(balanced_df, "BALANCED SUBSET 6000")

print("Shortages:")
display(pd.DataFrame(balanced_shortages))

BALANCED SUBSET 6000
Total: 6000

Por split:


,count
split,
train,4800
validation,600
test,600



Por sentimiento:


,count
sentiment_label,
positive,2000
neutral,2000
negative,2000



Split x sentimiento:


sentiment_label,negative,neutral,positive
split,,,
test,200,200,200
train,1600,1600,1600
validation,200,200,200



Food rich score:


,food_rich_score
count,6000.000000
mean,11.805500
std,7.360346
min,1.000000
25%,7.000000
50%,10.000000
75%,15.000000
max,59.000000


Shortages:


""


In [13]:
# ============================================================
# 12. Crear food-rich subset 2.000
# ============================================================

# Nos quedamos con reseñas con señales gastronómicas claras.
foodrich_pool = df[df["food_rich_score"] >= 8].copy()

print("Food-rich pool:", len(foodrich_pool))

# 2.001 aprox:
# - 667 por clase
# - split aproximado por clase: 533 train, 67 validation, 67 test

foodrich_plan = {
    "train": {
        "positive": 533,
        "neutral": 533,
        "negative": 533,
    },
    "validation": {
        "positive": 67,
        "neutral": 67,
        "negative": 67,
    },
    "test": {
        "positive": 67,
        "neutral": 67,
        "negative": 67,
    },
}

foodrich_df, foodrich_shortages = sample_by_plan(
    foodrich_pool,
    foodrich_plan,
    sort_by="food_rich_score",
    ascending=False
)

foodrich_df["subset_foodrich_2000"] = True

show_subset_summary(foodrich_df, "FOOD-RICH SUBSET 2000")

print("Shortages:")
display(pd.DataFrame(foodrich_shortages))

Food-rich pool: 55906
FOOD-RICH SUBSET 2000
Total: 2001

Por split:


,count
split,
train,1599
validation,201
test,201



Por sentimiento:


,count
sentiment_label,
positive,667
neutral,667
negative,667



Split x sentimiento:


sentiment_label,negative,neutral,positive
split,,,
test,67,67,67
train,533,533,533
validation,67,67,67



Food rich score:


,food_rich_score
count,2001.000000
mean,34.546227
std,7.610518
min,24.000000
25%,28.000000
50%,34.000000
75%,38.000000
max,88.000000


Shortages:


""


In [14]:
# ============================================================
# 13. Crear gold seed subset 300
# ============================================================

# Este será el primer archivo que conviene traducir y revisar.
# 300 reviews:
# - 100 por clase
# - 80 train, 10 validation, 10 test por clase
# - prioriza reseñas food-rich.

gold_pool = foodrich_pool.copy()

gold_plan = {
    "train": {
        "positive": 80,
        "neutral": 80,
        "negative": 80,
    },
    "validation": {
        "positive": 10,
        "neutral": 10,
        "negative": 10,
    },
    "test": {
        "positive": 10,
        "neutral": 10,
        "negative": 10,
    },
}

gold_seed_df, gold_shortages = sample_by_plan(
    gold_pool,
    gold_plan,
    sort_by="food_rich_score",
    ascending=False
)

gold_seed_df["subset_gold_seed_300"] = True

show_subset_summary(gold_seed_df, "GOLD SEED SUBSET 300")

print("Shortages:")
display(pd.DataFrame(gold_shortages))

GOLD SEED SUBSET 300
Total: 300

Por split:


,count
split,
train,240
validation,30
test,30



Por sentimiento:


,count
sentiment_label,
positive,100
neutral,100
negative,100



Split x sentimiento:


sentiment_label,negative,neutral,positive
split,,,
test,10,10,10
train,80,80,80
validation,10,10,10



Food rich score:


,food_rich_score
count,300.000000
mean,46.040000
std,8.242057
min,36.000000
25%,40.000000
50%,45.000000
75%,50.000000
max,88.000000


Shortages:


""


In [15]:
# ============================================================
# 14. Construir formato de traducción
# ============================================================

def build_translation_document_id(corpus_document_id: str) -> str:
    digest = hashlib.sha256(str(corpus_document_id).encode("utf-8")).hexdigest()[:16]
    return f"tr_yelp_{digest}"


def to_translation_format(subset_df: pd.DataFrame, subset_name: str) -> pd.DataFrame:
    out = subset_df.copy()

    out["translation_document_id"] = out["corpus_document_id"].apply(build_translation_document_id)
    out["source_language"] = "en"
    out["target_language"] = "es"
    out["text_es"] = ""
    out["translation_status"] = "pending"
    out["translation_notes"] = ""
    out["subset_name"] = subset_name

    columns = [
        "translation_document_id",
        "subset_name",
        "corpus_document_id",
        "source_system_code",
        "source_dataset",
        "source_review_id",
        "source_business_id",
        "source_user_id",
        "business_name",
        "city",
        "state",
        "rating_value",
        "sentiment_label",
        "split",
        "language",
        "source_language",
        "target_language",
        "text_en",
        "text_en_normalized",
        "text_es",
        "translation_status",
        "translation_notes",
        "food_category_tags",
        "candidate_food_terms_found",
        "candidate_food_term_count",
        "food_rich_score",
        "review_date",
        "task_scope",
        "is_training_eligible",
        "quality_flags",
        "source_metrics",
        "created_at",
    ]

    existing_columns = [col for col in columns if col in out.columns]
    return out[existing_columns].copy()


balanced_translation_df = to_translation_format(balanced_df, "balanced_6000")
foodrich_translation_df = to_translation_format(foodrich_df, "foodrich_2000")
gold_translation_df = to_translation_format(gold_seed_df, "gold_seed_300")

display(gold_translation_df.head(3))

,translation_document_id,subset_name,corpus_document_id,source_system_code,source_dataset,source_review_id,source_business_id,source_user_id,business_name,city,state,rating_value,sentiment_label,split,language,source_language,target_language,text_en,text_en_normalized,text_es,translation_status,translation_notes,food_category_tags,candidate_food_terms_found,candidate_food_term_count,food_rich_score,review_date,task_scope,is_training_eligible,quality_flags,source_metrics,created_at
0,tr_yelp_bc72f132b3b1ae58,gold_seed_300,yelp_df377137ab1067b74ef165aa,yelp_open_dataset,yelp_open_dataset,PGafLTkRukU9gV7bUv1VMg,5iuo1kvv0XZMS0bUOoLz2Q,d5uwEdhzpGVOqx_Zi7DoRQ,Bistro St. Tropez,Philadelphia,PA,5.0,positive,train,en,en,es,"This review is for their restaurant week selection. WOW! So good. 4 course meal (plus bread) for $35 each, plus tax and tip. If I had to do it again, I'd choose the Tarte de Legumes, the Saumon, and the Tart Tatin. Also, good-sized portions on ev...","This review is for their restaurant week selection. WOW! So good. 4 course meal (plus bread) for $35 each, plus tax and tip. If I had to do it again, I'd choose the Tarte de Legumes, the Saumon, and the Tart Tatin. Also, good-sized portions on ev...",,pending,,"[French, Restaurants]","[appetizer, cake, calamari, dessert, dishes, eggplant, entree, food, ice cream, main course, meal, menu, salad, salmon, seafood, soup, soups, steak]",18,88,2012-02-01 04:18:14,"[dish_extraction, review_sentiment, dish_sentiment, recommendation_signal]",True,"{'has_text': True, 'has_rating': True, 'has_business_metadata': True, 'text_length_chars': 4944, 'label_is_weak': True, 'label_source': 'rating_value'}","{'useful_count': 3, 'funny_count': 0, 'cool_count': 2}",2026-05-06T10:58:03.731497+00:00
1,tr_yelp_8e893eed7c9e70fa,gold_seed_300,yelp_968f332330ac270aaa868df6,yelp_open_dataset,yelp_open_dataset,4aBWpF-e7wf4Un2v8iSSNw,pym7c6ZFEtmoH16xN2ApBg,rL5uMarIYnEMgDr09pafqg,Katie's Restaurant & Bar,New Orleans,LA,5.0,positive,train,en,en,es,"I have been patronizing Katie's since early 1999, back in my days of being employed by the City of NOLA. It is one of my absolute favorite restaurants in the city and has never left me anything but completely satiated after every visit. It is als...","I have been patronizing Katie's since early 1999, back in my days of being employed by the City of NOLA. It is one of my absolute favorite restaurants in the city and has never left me anything but completely satiated after every visit. It is als...",,pending,,"[American (Traditional), Bars, Restaurants, Seafood]","[appetizers, bacon, bbq, brunch, cake, crab, crab cake, dishes, eggplant, eggs, food, fries, menu, oysters, pasta, pizza, pizzas, sandwich, sandwiches, seafood, shrimp]",21,87,2015-01-20 16:14:09,"[dish_extraction, review_sentiment, dish_sentiment, recommendation_signal]",True,"{'has_text': True, 'has_rating': True, 'has_business_metadata': True, 'text_length_chars': 4943, 'label_is_weak': True, 'label_source': 'rating_value'}","{'useful_count': 16, 'funny_count': 8, 'cool_count': 10}",2026-05-06T10:58:05.806341+00:00
2,tr_yelp_7361ac4dfe9c5590,gold_seed_300,yelp_2ba1323a0c0bef272a6fad98,yelp_open_dataset,yelp_open_dataset,kHFdHaq9HStlHWjhEHQXHQ,oBhJuukGRqPVvYBfTkhuZA,g8hEl1POjEzTaM2YlCiHUg,Square 1682,Philadelphia,PA,5.0,positive,train,en,en,es,RW review: I had the BEST time last night at Square 1682 for RW festivities! We had a reservation for 7:30pm for party of 6. My husband and I arrived a few minutes early and were greeted by a sleek hostess immediately and even took our coats. We ...,RW review: I had the BEST time last night at Square 1682 for RW festivities! We had a reservation for 7:30pm for party of 6. My husband and I arrived a few minutes early and were greeted by a sleek hostess immediately and even took our coats. We ...,,pending,,"[American (New), Bars, Breakfast & Brunch, Restaurants]","[appetizer, appetizers, cake, dessert, dish, entree, food, gnocchi, ice cream, lobster, meal, me

In [16]:
# ============================================================
# 15. Combinar subconjuntos únicos
# ============================================================

all_parts = [
    balanced_translation_df,
    foodrich_translation_df,
    gold_translation_df,
]

combined_raw = pd.concat(all_parts, ignore_index=True)

# Agrupar memberships por documento
membership_map = (
    combined_raw
    .groupby("translation_document_id")["subset_name"]
    .apply(lambda values: sorted(set(values)))
    .to_dict()
)

# Quedarnos con una fila por documento
combined_unique_df = (
    combined_raw
    .sort_values(["subset_name", "translation_document_id"])
    .drop_duplicates(subset=["translation_document_id"])
    .copy()
)

combined_unique_df["subset_membership"] = combined_unique_df["translation_document_id"].map(membership_map)

# Prioridad de traducción:
# gold_seed > foodrich > balanced
def compute_translation_priority(memberships):
    memberships = set(memberships)
    if "gold_seed_300" in memberships:
        return 3
    if "foodrich_2000" in memberships:
        return 2
    return 1

combined_unique_df["translation_priority"] = combined_unique_df["subset_membership"].apply(compute_translation_priority)

combined_unique_df = combined_unique_df.sort_values(
    ["translation_priority", "sentiment_label", "split", "food_rich_score"],
    ascending=[False, True, True, False]
).reset_index(drop=True)

print("Total balanced:", len(balanced_translation_df))
print("Total foodrich:", len(foodrich_translation_df))
print("Total gold:", len(gold_translation_df))
print("Total único combinado:", len(combined_unique_df))

display(
    combined_unique_df["translation_priority"].value_counts().sort_index()
)

display(combined_unique_df.head(5))

Total balanced: 6000
Total foodrich: 2001
Total gold: 300
Total único combinado: 7749


,count
translation_priority,
1,5748
2,1701
3,300


,translation_document_id,subset_name,corpus_document_id,source_system_code,source_dataset,source_review_id,source_business_id,source_user_id,business_name,city,state,rating_value,sentiment_label,split,language,source_language,target_language,text_en,text_en_normalized,text_es,translation_status,translation_notes,food_category_tags,candidate_food_terms_found,candidate_food_term_count,food_rich_score,review_date,task_scope,is_training_eligible,quality_flags,source_metrics,created_at,subset_membership,translation_priority
0,tr_yelp_55ed3ce3deba500c,foodrich_2000,yelp_c906f58fc054fc1c6a560d96,yelp_open_dataset,yelp_open_dataset,Btlxy41zkC7uudwwUdVEXA,ySXKjndttZjNy3kcqRqG3g,Vqe01M8MkBXA3eCBUWZpQg,Islamorada Fish Company Restaurant,Tampa,FL,2.0,negative,test,en,en,es,"A disappointment to say the least. The food is not bad, just not what I hoped for, which was a great seafood restaurant in Brandon, FL. The wait staff is lacking, like many of the other posts I think they hired absolutely no one with prior wait e...","A disappointment to say the least. The food is not bad, just not what I hoped for, which was a great seafood restaurant in Brandon, FL. The wait staff is lacking, like many of the other posts I think they hired absolutely no one with prior wait e...",,pending,,"[Restaurants, Sandwiches]","[appetizers, calamari, dish, dishes, entree, food, marinara, menu, oysters, pasta, seafood, shrimp, spaghetti]",13,51,2015-08-19 21:03:22,"[dish_extraction, review_sentiment, dish_sentiment, recommendation_signal]",True,"{'has_text': True, 'has_rating': True, 'has_business_metadata': True, 'text_length_chars': 2542, 'label_is_weak': True, 'label_source': 'rating_value'}","{'useful_count': 9, 'funny_count': 3, 'cool_count': 2}",2026-05-06T10:57:58.361847+00:00,"[foodrich_2000, gold_seed_300]",3
1,tr_yelp_c12ef107d086cf59,foodrich_2000,yelp_fb1ec98aca611308546832fc,yelp_open_dataset,yelp_open_dataset,FkNx1YF8T5H4o3nVuj74vA,E1gf1YIWOo1BgzMUwJtEZg,SCUFKNjzzOfFMMGYSZYyog,Top Tomato Bar & Pizza,Philadelphia,PA,2.0,negative,test,en,en,es,"I gave this place 2 stars instead of 1 b/c they allowed me twice to use a Living Social deal for takeout, which I really appreciated. However, they screwed up the orders both times - the first time wasn't a big deal; they just put some lettuce on...","I gave this place 2 stars instead of 1 b/c they allowed me twice to use a Living Social deal for takeout, which I really appreciated. However, they screwed up the orders both times - the first time wasn't a big deal; they just put some lettuce on...",,pending,,"[Bars, Breakfast & Brunch, Cocktail Bars, Food, Food Delivery Services, Italian, Pizza, Restaurants, Sandwiches, Sports Bars]","[food, meal, menu, pasta, pizza, salad, salmon, sandwich, sandwiches, shrimp]",10,49,2014-05-27 17:19:31,"[dish_extraction, review_sentiment, dish_sentiment, recommendation_signal]",True,"{'has_text': True, 'has_rating': True, 'has_business_metadata': True, 'text_length_chars': 1772, 'label_is_weak': True, 'label_source': 'rating_value'}","{'useful_count': 0, 'funny_count': 0, 'cool_count': 0}",2026-05-06T10:58:03.048932+00:00,"[foodrich_2000, gold_seed_300]",3
2,tr_yelp_8af4c67217a4ae4b,balanced_6000,yelp_73206146b5a419119719d729,yelp_open_dataset,yelp_open_dataset,Iw4ux8a7EdwZF8N8jk60ww,WecgAHgzAPLOmM-6-Iga2A,E5q2NS6tFy7kvom0qL3fFw,Cafe Roma,New Orleans,LA,1.0,negative,test,en,en,es,"My girlfriend and I stopped there around 11PM on a Saturday night. The place was completely dead and we were the only people in the restaurant, so we figured we would get a better than average meal and service, based on many of the yelp reviewers...","My girlfriend and I stopped there around 11PM on a Saturday night. The place was completely dead and we were the only people in the restaurant, so we figured we would get a better than average meal and service, based on many of the yelp reviewers...",,pending,,"[Italian, Pizza, Restaurants, Sandwiches]","[food, hummus, main course, marinara, meal, past

In [17]:
# ============================================================
# 16. Guardar archivos JSONL
# ============================================================

def make_json_serializable(value):
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, float) and np.isnan(value):
        return None
    return value


def save_jsonl(df_to_save: pd.DataFrame, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("w", encoding="utf-8") as f:
        for record in df_to_save.to_dict(orient="records"):
            clean_record = {
                str(key): make_json_serializable(value)
                for key, value in record.items()
            }
            f.write(json.dumps(clean_record, ensure_ascii=False) + "\n")

    print(f"Guardado: {path} ({len(df_to_save):,} registros)")


balanced_path = OUTPUT_DIR / "yelp_translation_balanced_6000.jsonl"
foodrich_path = OUTPUT_DIR / "yelp_translation_foodrich_2000.jsonl"
gold_path = OUTPUT_DIR / "yelp_translation_gold_seed_300.jsonl"
combined_path = OUTPUT_DIR / "yelp_translation_all_unique.jsonl"

save_jsonl(balanced_translation_df, balanced_path)
save_jsonl(foodrich_translation_df, foodrich_path)
save_jsonl(gold_translation_df, gold_path)
save_jsonl(combined_unique_df, combined_path)

Guardado: /content/hidden_gems_ai/outputs/translation_subsets/yelp_translation_balanced_6000.jsonl (6,000 registros)
Guardado: /content/hidden_gems_ai/outputs/translation_subsets/yelp_translation_foodrich_2000.jsonl (2,001 registros)
Guardado: /content/hidden_gems_ai/outputs/translation_subsets/yelp_translation_gold_seed_300.jsonl (300 registros)
Guardado: /content/hidden_gems_ai/outputs/translation_subsets/yelp_translation_all_unique.jsonl (7,749 registros)


In [18]:
# ============================================================
# 17. Crear chunks para traducción
# ============================================================

CHUNK_SIZE = 100

chunks_output_dir = CHUNKS_DIR / "all_unique_chunks"
chunks_output_dir.mkdir(parents=True, exist_ok=True)

# Limpiar chunks antiguos
for old_file in chunks_output_dir.glob("*.jsonl"):
    old_file.unlink()

chunk_paths = []

for chunk_index, start in enumerate(range(0, len(combined_unique_df), CHUNK_SIZE), start=1):
    chunk_df = combined_unique_df.iloc[start:start + CHUNK_SIZE].copy()

    chunk_path = chunks_output_dir / f"yelp_translation_chunk_{chunk_index:03d}.jsonl"
    save_jsonl(chunk_df, chunk_path)

    chunk_paths.append(str(chunk_path))

print(f"Chunks creados: {len(chunk_paths)}")
print("Primer chunk:")
print(chunk_paths[0])

Guardado: /content/hidden_gems_ai/outputs/translation_subsets/chunks_for_translation/all_unique_chunks/yelp_translation_chunk_001.jsonl (100 registros)
Guardado: /content/hidden_gems_ai/outputs/translation_subsets/chunks_for_translation/all_unique_chunks/yelp_translation_chunk_002.jsonl (100 registros)
Guardado: /content/hidden_gems_ai/outputs/translation_subsets/chunks_for_translation/all_unique_chunks/yelp_translation_chunk_003.jsonl (100 registros)
Guardado: /content/hidden_gems_ai/outputs/translation_subsets/chunks_for_translation/all_unique_chunks/yelp_translation_chunk_004.jsonl (100 registros)
Guardado: /content/hidden_gems_ai/outputs/translation_subsets/chunks_for_translation/all_unique_chunks/yelp_translation_chunk_005.jsonl (100 registros)
Guardado: /content/hidden_gems_ai/outputs/translation_subsets/chunks_for_translation/all_unique_chunks/yelp_translation_chunk_006.jsonl (100 registros)
Guardado: /content/hidden_gems_ai/outputs/translation_subsets/chunks_for_translation/all

In [19]:
# ============================================================
# 18. Guardar resumen de subconjuntos
# ============================================================

def subset_counts(subset_df: pd.DataFrame) -> dict:
    return {
        "total": int(len(subset_df)),
        "split_counts": {
            str(k): int(v)
            for k, v in subset_df["split"].value_counts().to_dict().items()
        },
        "sentiment_counts": {
            str(k): int(v)
            for k, v in subset_df["sentiment_label"].value_counts().to_dict().items()
        },
        "split_x_sentiment": {
            str(split): {
                str(label): int(value)
                for label, value in row.items()
            }
            for split, row in pd.crosstab(
                subset_df["split"],
                subset_df["sentiment_label"]
            ).to_dict(orient="index").items()
        },
        "food_rich_score": {
            "min": float(subset_df["food_rich_score"].min()),
            "max": float(subset_df["food_rich_score"].max()),
            "mean": float(subset_df["food_rich_score"].mean()),
            "median": float(subset_df["food_rich_score"].median()),
        }
    }


summary = {
    "created_for": "Hidden Gems - Spanish adaptation subset builder",
    "source_corpus_path": str(CORPUS_PATH),
    "outputs": {
        "balanced_6000": str(balanced_path),
        "foodrich_2000": str(foodrich_path),
        "gold_seed_300": str(gold_path),
        "all_unique": str(combined_path),
        "chunks_dir": str(chunks_output_dir),
        "chunk_size": CHUNK_SIZE,
        "chunk_count": len(chunk_paths),
    },
    "subsets": {
        "balanced_6000": subset_counts(balanced_translation_df),
        "foodrich_2000": subset_counts(foodrich_translation_df),
        "gold_seed_300": subset_counts(gold_translation_df),
        "all_unique": subset_counts(combined_unique_df),
    },
    "shortages": {
        "balanced": balanced_shortages,
        "foodrich": foodrich_shortages,
        "gold_seed": gold_shortages,
    }
}

summary_path = OUTPUT_DIR / "yelp_translation_subset_summary.json"

with summary_path.open("w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print("Resumen guardado en:")
print(summary_path)

print(json.dumps(summary, indent=2, ensure_ascii=False)[:3000])

Resumen guardado en:
/content/hidden_gems_ai/outputs/translation_subsets/yelp_translation_subset_summary.json
{
  "created_for": "Hidden Gems - Spanish adaptation subset builder",
  "source_corpus_path": "/content/hidden_gems_ai/data/yelp_food_reviews_corpus_sample_100k_lines.jsonl",
  "outputs": {
    "balanced_6000": "/content/hidden_gems_ai/outputs/translation_subsets/yelp_translation_balanced_6000.jsonl",
    "foodrich_2000": "/content/hidden_gems_ai/outputs/translation_subsets/yelp_translation_foodrich_2000.jsonl",
    "gold_seed_300": "/content/hidden_gems_ai/outputs/translation_subsets/yelp_translation_gold_seed_300.jsonl",
    "all_unique": "/content/hidden_gems_ai/outputs/translation_subsets/yelp_translation_all_unique.jsonl",
    "chunks_dir": "/content/hidden_gems_ai/outputs/translation_subsets/chunks_for_translation/all_unique_chunks",
    "chunk_size": 100,
    "chunk_count": 78
  },
  "subsets": {
    "balanced_6000": {
      "total": 6000,
      "split_counts": {
       

In [20]:
# ============================================================
# 19. Comprobaciones finales
# ============================================================

checks = {
    "balanced_has_rows": len(balanced_translation_df) > 0,
    "foodrich_has_rows": len(foodrich_translation_df) > 0,
    "gold_has_rows": len(gold_translation_df) > 0,
    "combined_has_rows": len(combined_unique_df) > 0,
    "combined_translation_ids_unique": combined_unique_df["translation_document_id"].is_unique,
    "gold_translation_ids_unique": gold_translation_df["translation_document_id"].is_unique,
    "all_have_source_text": combined_unique_df["text_en"].notna().all(),
    "all_text_es_empty": (combined_unique_df["text_es"] == "").all(),
    "balanced_file_exists": balanced_path.exists(),
    "foodrich_file_exists": foodrich_path.exists(),
    "gold_file_exists": gold_path.exists(),
    "combined_file_exists": combined_path.exists(),
    "summary_file_exists": summary_path.exists(),
    "chunks_created": len(chunk_paths) > 0,
}

for check_name, value in checks.items():
    print(f"{check_name}: {value}")

if not all(checks.values()):
    raise ValueError("Alguna comprobación final ha fallado.")

print("\nSubconjuntos de traducción generados correctamente.")

balanced_has_rows: True
foodrich_has_rows: True
gold_has_rows: True
combined_has_rows: True
combined_translation_ids_unique: True
gold_translation_ids_unique: True
all_have_source_text: True
all_text_es_empty: True
balanced_file_exists: True
foodrich_file_exists: True
gold_file_exists: True
combined_file_exists: True
summary_file_exists: True
chunks_created: True

Subconjuntos de traducción generados correctamente.


## 20. Traducción del Gold Seed 300 al español

En este bloque se traduce el subconjunto `gold_seed_300` desde inglés a español.

El objetivo no es crear una traducción definitiva perfecta, sino generar un corpus puente en español para:

- adaptar modelos al idioma objetivo de Hidden Gems;
- comparar inglés frente a español traducido;
- preparar una primera base para entrenamiento y validación;
- acercar el trabajo NLP al caso real de Google Reviews Sevilla.

El archivo de salida conservará el texto original en `text_en` y añadirá la traducción en `text_es`.

In [21]:
# ============================================================
# 20. Instalar dependencias de traducción
# ============================================================

!pip -q install transformers sentencepiece sacremoses accelerate tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 43.4 MB/s eta 0:00:00


In [22]:
# ============================================================
# 21. Cargar modelo de traducción EN -> ES
# ============================================================

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm.auto import tqdm
import json
import re
from pathlib import Path
import pandas as pd
import numpy as np

TRANSLATION_MODEL_NAME = "Helsinki-NLP/opus-mt-en-es"

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Dispositivo:", device)
print("Modelo:", TRANSLATION_MODEL_NAME)

tokenizer = AutoTokenizer.from_pretrained(TRANSLATION_MODEL_NAME)
translation_model = AutoModelForSeq2SeqLM.from_pretrained(TRANSLATION_MODEL_NAME).to(device)

print("Modelo de traducción cargado correctamente.")

Dispositivo: cuda
Modelo: Helsinki-NLP/opus-mt-en-es


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/826k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/312M [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

Modelo de traducción cargado correctamente.


In [23]:
# ============================================================
# 22. Cargar Gold Seed 300
# ============================================================

gold_path = OUTPUT_DIR / "yelp_translation_gold_seed_300.jsonl"

if not gold_path.exists():
    raise FileNotFoundError(f"No se ha encontrado el archivo gold seed en: {gold_path}")

gold_records = []

with gold_path.open("r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            gold_records.append(json.loads(line))

gold_df = pd.DataFrame(gold_records)

print("Registros Gold Seed:", len(gold_df))
display(gold_df[["translation_document_id", "sentiment_label", "split", "rating_value", "food_rich_score", "text_en", "text_es"]].head(3))

print("\nDistribución sentimiento:")
display(gold_df["sentiment_label"].value_counts())

print("\nDistribución split:")
display(gold_df["split"].value_counts())

Registros Gold Seed: 300


,translation_document_id,sentiment_label,split,rating_value,food_rich_score,text_en,text_es
0,tr_yelp_bc72f132b3b1ae58,positive,train,5.0,88,"This review is for their restaurant week selection. WOW! So good. 4 course meal (plus bread) for $35 each, plus tax and tip. If I had to do it again, I'd choose the Tarte de Legumes, the Saumon, and the Tart Tatin. Also, good-sized portions on ev...",
1,tr_yelp_8e893eed7c9e70fa,positive,train,5.0,87,"I have been patronizing Katie's since early 1999, back in my days of being employed by the City of NOLA. It is one of my absolute favorite restaurants in the city and has never left me anything but completely satiated after every visit. It is als...",
2,tr_yelp_7361ac4dfe9c5590,positive,train,5.0,87,RW review: I had the BEST time last night at Square 1682 for RW festivities! We had a reservation for 7:30pm for party of 6. My husband and I arrived a few minutes early and were greeted by a sleek hostess immediately and even took our coats. We ...,



Distribución sentimiento:


,count
sentiment_label,
positive,100
neutral,100
negative,100



Distribución split:


,count
split,
train,240
validation,30
test,30


In [24]:
# ============================================================
# 23. Funciones robustas de traducción
# ============================================================

def split_text_for_translation(text: str, max_chars: int = 850) -> list[str]:
    """
    Divide textos largos en fragmentos razonables para traducción.
    Se intenta cortar por frases para evitar perder contexto.
    """

    if not isinstance(text, str):
        return []

    text = re.sub(r"\s+", " ", text).strip()

    if len(text) <= max_chars:
        return [text]

    # División básica por frases
    sentences = re.split(r"(?<=[.!?])\s+", text)

    chunks = []
    current = ""

    for sentence in sentences:
        sentence = sentence.strip()
        if not sentence:
            continue

        if len(current) + len(sentence) + 1 <= max_chars:
            current = f"{current} {sentence}".strip()
        else:
            if current:
                chunks.append(current)

            # Si una frase es demasiado larga, se corta por longitud
            if len(sentence) > max_chars:
                for i in range(0, len(sentence), max_chars):
                    chunks.append(sentence[i:i + max_chars].strip())
                current = ""
            else:
                current = sentence

    if current:
        chunks.append(current)

    return chunks


def translate_batch_texts(texts: list[str], batch_size: int = 8, max_new_tokens: int = 512) -> list[str]:
    """
    Traduce una lista de textos cortos/fragments.
    """

    translations = []

    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]

        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        ).to(device)

        with torch.no_grad():
            outputs = translation_model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                num_beams=4,
                early_stopping=True
            )

        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        translations.extend(decoded)

    return translations


def translate_long_review(text: str) -> str:
    """
    Traduce una review completa dividiéndola en fragmentos si es necesario.
    """

    chunks = split_text_for_translation(text)

    if not chunks:
        return ""

    translated_chunks = translate_batch_texts(chunks, batch_size=8)

    translated_text = " ".join(translated_chunks)
    translated_text = re.sub(r"\s+", " ", translated_text).strip()

    return translated_text


# Prueba rápida
sample_text = "The burger was amazing, juicy and perfectly cooked. The fries were crispy, but the service was slow."
print("Original:")
print(sample_text)

print("\nTraducción:")
print(translate_long_review(sample_text))

Original:
The burger was amazing, juicy and perfectly cooked. The fries were crispy, but the service was slow.

Traducción:
La hamburguesa era increíble, jugosa y perfectamente cocida. Las papas fritas eran crujientes, pero el servicio era lento.


In [25]:
# ============================================================
# 24. Traducir Gold Seed 300
# ============================================================

translated_records = []

checkpoint_dir = OUTPUT_DIR / "translation_checkpoints"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

checkpoint_path = checkpoint_dir / "gold_seed_300_translation_checkpoint.jsonl"

# Si existe checkpoint previo, lo cargamos para no repetir trabajo
already_translated = {}

if checkpoint_path.exists():
    with checkpoint_path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                record = json.loads(line)
                already_translated[record["translation_document_id"]] = record

    print(f"Checkpoint cargado: {len(already_translated)} registros traducidos.")

for idx, row in tqdm(gold_df.iterrows(), total=len(gold_df), desc="Traduciendo Gold Seed"):
    translation_document_id = row["translation_document_id"]

    if translation_document_id in already_translated:
        translated_records.append(already_translated[translation_document_id])
        continue

    record = row.to_dict()

    try:
        text_en = record.get("text_en", "")
        text_es = translate_long_review(text_en)

        record["text_es"] = text_es
        record["translation_status"] = "machine_translated"
        record["translation_model"] = TRANSLATION_MODEL_NAME
        record["translation_method"] = "huggingface_marianmt"
        record["translation_notes"] = "Traducción automática EN-ES pendiente de revisión manual."
        record["translation_error"] = ""

    except Exception as e:
        record["text_es"] = ""
        record["translation_status"] = "translation_error"
        record["translation_model"] = TRANSLATION_MODEL_NAME
        record["translation_method"] = "huggingface_marianmt"
        record["translation_notes"] = "Error durante la traducción automática."
        record["translation_error"] = str(e)

    translated_records.append(record)

    # Guardar checkpoint cada 10 registros
    if len(translated_records) % 10 == 0:
        with checkpoint_path.open("w", encoding="utf-8") as f:
            for r in translated_records:
                f.write(json.dumps(r, ensure_ascii=False) + "\n")

# Guardado final de checkpoint
with checkpoint_path.open("w", encoding="utf-8") as f:
    for r in translated_records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

translated_gold_df = pd.DataFrame(translated_records)

print("Traducción completada.")
print("Registros traducidos:", len(translated_gold_df))
display(translated_gold_df[["sentiment_label", "split", "translation_status", "text_en", "text_es"]].head(5))

Traduciendo Gold Seed:   0%|          | 0/300 [00:00<?, ?it/s]

Traducción completada.
Registros traducidos: 300


,sentiment_label,split,translation_status,text_en,text_es
0,positive,train,machine_translated,"This review is for their restaurant week selection. WOW! So good. 4 course meal (plus bread) for $35 each, plus tax and tip. If I had to do it again, I'd choose the Tarte de Legumes, the Saumon, and the Tart Tatin. Also, good-sized portions on ev...","Esta revisión es para su selección de la semana de restaurante. WOW! Tan bueno. 4 platos (más pan) por $35 cada uno, más impuestos y propina. Si tuviera que hacerlo de nuevo, elegiría la Tarte de Legumes, el Saumon, y el Tart Tatin. También, porc..."
1,positive,train,machine_translated,"I have been patronizing Katie's since early 1999, back in my days of being employed by the City of NOLA. It is one of my absolute favorite restaurants in the city and has never left me anything but completely satiated after every visit. It is als...","He sido condescendiente de Katie desde principios de 1999, en mis días de ser empleado por la ciudad de NOLA. Es uno de mis restaurantes favoritos absolutos en la ciudad y nunca me ha dejado nada más que completamente saciada después de cada visi..."
2,positive,train,machine_translated,RW review: I had the BEST time last night at Square 1682 for RW festivities! We had a reservation for 7:30pm for party of 6. My husband and I arrived a few minutes early and were greeted by a sleek hostess immediately and even took our coats. We ...,OR review: Yo tuve la mejor noche en la Plaza 1682 para las fiestas del OR! Tuvimos una reserva para las 7:30pm para la fiesta de 6. Mi marido y yo llegamos unos minutos antes y fuimos recibidos por una anfitriona elegante inmediatamente e inclus...
3,positive,train,machine_translated,"Khyber Pass Pub is located in a historic city, in a historic building, with a historic bar that came from the Centennial World's Fair if 1876. Khyber Pass itself, has only been open since 1970, making it a baby in comparison. Still, the place is ...","Khyber Pass Pub se encuentra en una ciudad histórica, en un edificio histórico, con un bar histórico que vino de la Feria del Mundo Centenario si 1876. Khyber Pass en sí, sólo ha estado abierto desde 1970, haciendo que sea un bebé en comparación...."
4,positive,train,machine_translated,"I have seen various reviews on Tavern. Some very negative. Others slightly positive. Most complimented the food but nothing spectacular. So, this past Friday night a few friends and I got together for dinner and I wanted to try some place new. I ...","He visto varias críticas en Tavern. Algunos muy negativos. Otros ligeramente positivos. La mayoría elogiaron la comida pero nada espectacular. Así, este pasado viernes por la noche unos amigos y me reuní para cenar y quería probar algún lugar nue..."


In [26]:
# ============================================================
# 25. QA básico de traducción
# ============================================================

translated_gold_df["text_es_length_chars"] = translated_gold_df["text_es"].astype(str).str.len()
translated_gold_df["text_en_length_chars"] = translated_gold_df["text_en"].astype(str).str.len()

qa_summary = {
    "total_records": int(len(translated_gold_df)),
    "translation_status_counts": {
        str(k): int(v)
        for k, v in translated_gold_df["translation_status"].value_counts().to_dict().items()
    },
    "empty_text_es_count": int((translated_gold_df["text_es_length_chars"] == 0).sum()),
    "text_es_length": {
        "min": int(translated_gold_df["text_es_length_chars"].min()),
        "max": int(translated_gold_df["text_es_length_chars"].max()),
        "mean": float(translated_gold_df["text_es_length_chars"].mean()),
        "median": float(translated_gold_df["text_es_length_chars"].median()),
    },
    "sentiment_counts": {
        str(k): int(v)
        for k, v in translated_gold_df["sentiment_label"].value_counts().to_dict().items()
    },
    "split_counts": {
        str(k): int(v)
        for k, v in translated_gold_df["split"].value_counts().to_dict().items()
    },
}

print(json.dumps(qa_summary, indent=2, ensure_ascii=False))

print("\nEjemplos aleatorios:")
sample_cols = [
    "translation_document_id",
    "sentiment_label",
    "rating_value",
    "text_en",
    "text_es",
    "candidate_food_terms_found"
]

display(
    translated_gold_df[sample_cols]
    .sample(5, random_state=42)
)

{
  "total_records": 300,
  "translation_status_counts": {
    "machine_translated": 300
  },
  "empty_text_es_count": 0,
  "text_es_length": {
    "min": 594,
    "max": 5343,
    "mean": 2637.47,
    "median": 2435.0
  },
  "sentiment_counts": {
    "positive": 100,
    "neutral": 100,
    "negative": 100
  },
  "split_counts": {
    "train": 240,
    "validation": 30,
    "test": 30
  }
}

Ejemplos aleatorios:


,translation_document_id,sentiment_label,rating_value,text_en,text_es,candidate_food_terms_found
203,tr_yelp_57930a40ddaa8407,negative,2.0,"They have dulled like their steak knives. Used to be a treat to go here and now it is just really expensive. Their target audience is clearly only people over the age of 60. There wasn't anything special about the appetizers or desserts, and unfo...","Se han embotado como sus cuchillos de bistec. Solía ser una delicia para ir aquí y ahora es realmente caro. Su público objetivo es claramente sólo las personas de más de 60 años. No había nada especial acerca de los aperitivos o postres, y desafo...","[appetizers, cake, crab, crab cake, dessert, desserts, dish, lobster, menu, salmon, steak]"
266,tr_yelp_3cf96b46c263e787,negative,2.0,I went here based on a lot of the hype surrounding the place. In some ways it didn't disappoint BUT in ways that I feel really mater it did! What I really liked about the place was the fact that it was located on the 37th Floor and had great view...,Fui aquí basado en una gran cantidad de la bombona que rodeaba el lugar. De alguna manera no decepcionó PERO en maneras que me siento realmente mater lo hizo! Lo que realmente me gustó sobre el lugar fue el hecho de que estaba situado en el piso ...,"[appetizer, entree, nachos, ribeye, ribs, salad, soup]"
152,tr_yelp_71b126876b8fcc20,neutral,3.0,"Tried Cha Cha Chow's food truck today, which is brand new to the burgeoning STL street food scene. Note to Yelp: You need to add the ability to add and track mobile food! Anyway, Cha Cha Chow serves five different kinds of tacos ($3 each, or 2 pl...","El camión de comida de Cha Cha Chow probado hoy, que es nuevo en la escena de la creciente comida callejera de STL. Nota para Yelp: Usted necesita agregar la capacidad de agregar y rastrear la comida móvil! De todos modos, Cha Cha Chow sirve cinc...","[burgers, curry, food, guacamole, menu, salsa, sandwiches, taco, tacos]"
9,tr_yelp_b6f9e8822e34a18b,positive,4.0,"If you're looking for an affordable and filling sushi spot, Megu is a Japanese restaurant located in Cherry Hill, New Jersey in the Village Walk Shopping Center. Also, it is BYOB and does not charge a corking fee. As you walk through the doors of...","Si usted está buscando un lugar asequible y lleno de sushi, Megu es un restaurante japonés situado en Cherry Hill, Nueva Jersey en el Village Walk Shopping Center. Además, es BYOB y no cobra una tarifa de corcho. Al caminar a través de las puerta...","[appetizer, appetizers, crab, dessert, dumpling, dumplings, food, ice cream, lobster, main course, menu, salad, salmon, sashimi, shrimp, sushi, tea, tempura, tuna]"
233,tr_yelp_4ac5b518ef78652f,negative,1.0,"My friends and I were visiting Nola for the first time and decided to yelp a restaurant close by to our hotel. This place had pretty good ratings so we decided to give it a try. We were on a laid back vacation, we were all dressed quite casual. T...","Mis amigos y yo estábamos visitando Nola por primera vez y decidimos gritar un restaurante cerca de nuestro hotel. Este lugar tenía muy buenas calificaciones por lo que decidimos darle una oportunidad. Estábamos en unas vacaciones relajadas, todo...","[burger, burgers, food, fries, salad, shrimp, soup]"


In [27]:
# ============================================================
# 26. Guardar Gold Seed traducido
# ============================================================

translated_gold_path = OUTPUT_DIR / "yelp_translation_gold_seed_300_es.jsonl"
translated_gold_summary_path = OUTPUT_DIR / "yelp_translation_gold_seed_300_es_summary.json"

def make_json_serializable(value):
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, float) and np.isnan(value):
        return None
    return value

with translated_gold_path.open("w", encoding="utf-8") as f:
    for record in translated_gold_df.to_dict(orient="records"):
        clean_record = {
            str(key): make_json_serializable(value)
            for key, value in record.items()
        }
        f.write(json.dumps(clean_record, ensure_ascii=False) + "\n")

with translated_gold_summary_path.open("w", encoding="utf-8") as f:
    json.dump(qa_summary, f, indent=2, ensure_ascii=False)

print("Archivo traducido guardado en:")
print(translated_gold_path)

print("\nResumen guardado en:")
print(translated_gold_summary_path)

Archivo traducido guardado en:
/content/hidden_gems_ai/outputs/translation_subsets/yelp_translation_gold_seed_300_es.jsonl

Resumen guardado en:
/content/hidden_gems_ai/outputs/translation_subsets/yelp_translation_gold_seed_300_es_summary.json


In [28]:
# ============================================================
# 27. Descargar archivo traducido
# ============================================================

from google.colab import files

files.download(str(translated_gold_path))
files.download(str(translated_gold_summary_path))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>